# BHX

BHX 데이터셋은 3D mask 기반 데이터셋과 달리, **series / SOP instance 단위의 bounding box annotation CSV**를 중심으로 분석한다.

이 노트북의 목표는 다음과 같다.
- CSV 3종의 구조와 포함 관계를 정리한다.
- labelName / labelType 분포를 확인한다.
- bbox 크기, 비율, 분포를 요약한다.
- downstream 학습에 사용할 수 있도록 데이터셋 단위 통계를 정리한다.


In [ ]:
from __future__ import annotations

import ast
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

DATA_ROOT = Path("/home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/data/bhx")
NOTEBOOK_ROOT = Path("/home/jovyan/aicon-gamma-datavol-1/hjgoh/ich-vlm/notebook")

CSV_1 = DATA_ROOT / "1_Initial_Manual_Labeling.csv"
CSV_2 = DATA_ROOT / "2_Extrapolation_to_All_Series.csv"
CSV_3 = DATA_ROOT / "3_Extrapolation_to_Selected_Series.csv"
EXAMPLE_IMG = DATA_ROOT / "example_annotated_images.png"

CSV_PATHS = {
    "initial_manual": CSV_1,
    "all_series": CSV_2,
    "selected_series": CSV_3,
}

In [ ]:
def load_bhx_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])
    return df


def parse_bbox(value: str) -> dict[str, float]:
    data = ast.literal_eval(value)
    return {
        "x": float(data["x"]),
        "y": float(data["y"]),
        "width": float(data["width"]),
        "height": float(data["height"]),
    }


def enrich_bbox(df: pd.DataFrame) -> pd.DataFrame:
    bbox_df = df["data"].apply(parse_bbox).apply(pd.Series)
    out = pd.concat([df.reset_index(drop=True), bbox_df], axis=1)
    out["area"] = out["width"] * out["height"]
    out["aspect_ratio"] = out["width"] / out["height"]
    return out


dfs = {name: load_bhx_csv(path) for name, path in CSV_PATHS.items()}
for name, df in dfs.items():
    print(name, df.shape)
    display(df.head(3))


## 1. Basic Info

여기서는 CSV 규모, 고유 study / series / SOP 수, 그리고 3개 CSV의 포함 관계를 확인한다.


In [ ]:
summary_rows = []
for name, df in dfs.items():
    summary_rows.append({
        "dataset": name,
        "rows": len(df),
        "studies": df["StudyInstanceUID"].nunique(),
        "series": df["SeriesInstanceUID"].nunique(),
        "sop": df["SOPInstanceUID"].nunique(),
        "label_types": ", ".join(sorted(df["labelType"].dropna().unique())),
        "labels": ", ".join(sorted(df["labelName"].dropna().unique())),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

set_1 = set(map(tuple, dfs["initial_manual"][ ["SOPInstanceUID", "SeriesInstanceUID", "StudyInstanceUID", "data", "labelName", "labelType"] ].to_numpy()))
set_2 = set(map(tuple, dfs["all_series"][ ["SOPInstanceUID", "SeriesInstanceUID", "StudyInstanceUID", "data", "labelName", "labelType"] ].to_numpy()))
set_3 = set(map(tuple, dfs["selected_series"][ ["SOPInstanceUID", "SeriesInstanceUID", "StudyInstanceUID", "data", "labelName", "labelType"] ].to_numpy()))

print("1 -> 2 추가 행 수:", len(set_2 - set_1))
print("2 -> 3 제외 행 수:", len(set_2 - set_3))
print("3는 1과 2의 어떤 부분집합인지 비교 필요")


## 2. Label Distribution

class imbalance를 확인하고, CSV별로 labelName / labelType 분포를 비교한다.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 16))
axes = axes.ravel()

for i, (name, df) in enumerate(dfs.items()):
    label_counts = df["labelName"].value_counts().sort_values(ascending=False)
    type_counts = df["labelType"].value_counts().sort_values(ascending=False)
    label_counts.plot(kind="bar", ax=axes[i * 2], title=f"{name} - labelName")
    type_counts.plot(kind="bar", ax=axes[i * 2 + 1], title=f"{name} - labelType")

plt.tight_layout()
plt.show()


## 3. BBox Geometry

bbox의 width, height, area, aspect ratio 분포를 확인한다.


In [ ]:
bbox_tables = {name: enrich_bbox(df) for name, df in dfs.items()}

for name, df in bbox_tables.items():
    print(f"### {name}")
    display(df[["width", "height", "area", "aspect_ratio"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

fig, axes = plt.subplots(3, 4, figsize=(20, 18))
axes = axes.ravel()
for i, (name, df) in enumerate(bbox_tables.items()):
    df["width"].hist(ax=axes[i * 4], bins=50)
    axes[i * 4].set_title(f"{name} width")
    df["height"].hist(ax=axes[i * 4 + 1], bins=50)
    axes[i * 4 + 1].set_title(f"{name} height")
    df["area"].hist(ax=axes[i * 4 + 2], bins=50)
    axes[i * 4 + 2].set_title(f"{name} area")
    df["aspect_ratio"].replace([np.inf, -np.inf], np.nan).dropna().hist(ax=axes[i * 4 + 3], bins=50)
    axes[i * 4 + 3].set_title(f"{name} aspect ratio")

plt.tight_layout()
plt.show()


## 4. Series / Study Unit Analysis

각 study 안에 몇 개의 series가 있는지, series당 annotation이 얼마나 몰려 있는지 확인한다.


In [ ]:
for name, df in dfs.items():
    print(f"### {name}")
    study_series = df.groupby("StudyInstanceUID")["SeriesInstanceUID"].nunique().sort_values(ascending=False)
    series_counts = df.groupby("SeriesInstanceUID").size().sort_values(ascending=False)
    print("study 당 series 개수 요약")
    display(study_series.describe())
    print("series 당 annotation 개수 요약")
    display(series_counts.describe())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, df) in zip(axes, dfs.items()):
    df.groupby("SeriesInstanceUID").size().hist(ax=ax, bins=50)
    ax.set_title(f"{name} - annotations per series")
plt.tight_layout()
plt.show()


## 5. Visualization

원본 CT가 포함되어 있지 않으므로, 제공된 예시 annotated image를 먼저 확인하고 필요하면 별도 후속 셀에서 샘플링 로직을 붙인다.


In [ ]:
if EXAMPLE_IMG.exists():
    img = plt.imread(EXAMPLE_IMG)
    plt.figure(figsize=(16, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title("BHX example annotated images")
    plt.show()
else:
    print(f"Missing example image: {EXAMPLE_IMG}")


## 6. Summary TODO

- `1_Initial_Manual_Labeling.csv`를 baseline set으로 정리
- `2_Extrapolation_to_All_Series.csv`를 full coverage set으로 정리
- `3_Extrapolation_to_Selected_Series.csv`를 cleaner subset으로 정리
- class imbalance, bbox size, aspect ratio, series distribution을 표로 요약
- downstream 학습용 추천 split 전략을 추가
